# ECB FX Reference Rates Bronze Ingestion

Thin notebook entrypoint for ECB daily euro foreign exchange reference rates Bronze ingestion.

This notebook delegates request handling, Europe-local date-window resolution, CSV parsing, deduplication, and Delta merge logic to `src/lakehouse`.

Execution modes:

- `backfill`: explicit inclusive date range
- `incremental`: rolling lookback window ending on the latest completed `Europe/Berlin` day

The target table must already exist. Run `00_platform_setup_catalog_schema.ipynb` first.

In [ ]:
import sys
import uuid
from pathlib import Path


def add_repo_src_to_path() -> str:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        src_dir = candidate / "src"
        if (src_dir / "lakehouse" / "__init__.py").exists():
            src_dir_str = str(src_dir)
            if src_dir_str not in sys.path:
                sys.path.insert(0, src_dir_str)
            return src_dir_str

    raise RuntimeError("Could not locate the repository src directory from the current notebook path")


REPO_SRC = add_repo_src_to_path()

from lakehouse.pipelines.bronze import run_ecb_bronze_ingestion

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("quote_currencies", "USD")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "7")
dbutils.widgets.text("run_id", str(uuid.uuid4()))


In [ ]:
catalog = dbutils.widgets.get("catalog").strip() or "market_macro"
target_table = f"{catalog}.brz_macro.raw_ecb_fx_ref_rates_daily"

result = run_ecb_bronze_ingestion(
    spark=spark,
    raw_quote_currencies=dbutils.widgets.get("quote_currencies").strip(),
    mode=dbutils.widgets.get("mode").strip().lower(),
    start_date_raw=dbutils.widgets.get("start_date").strip(),
    end_date_raw=dbutils.widgets.get("end_date").strip(),
    lookback_days_raw=dbutils.widgets.get("lookback_days").strip(),
    run_id=dbutils.widgets.get("run_id").strip(),
    target_table=target_table,
    display_fn=display,
)

result.as_dict()
